In [2]:
import pandas as pd
import numpy as np
orders = pd.read_csv("olist_orders_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")

In [3]:
df = orders.merge(customers, on="customer_id")

df = df.merge(items, on="order_id")

df = df.merge(products, on="product_id")

df = df.merge(payments, on="order_id")

Revenue

In [6]:
df["Revenue"] = df["price"] + df["freight_value"]


Monthly Sales

In [11]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

monthly = (
    df.groupby(
        df["order_purchase_timestamp"].dt.to_period("M")
    )["Revenue"]
    .sum()
)

print(monthly)

order_purchase_timestamp
2016-09        211.29
2016-10      58550.03
2016-12         19.62
2017-01     146455.54
2017-02     302230.37
2017-03     457782.29
2017-04     448742.78
2017-05     630089.88
2017-06     526495.91
2017-07     627312.81
2017-08     700577.57
2017-09     763951.95
2017-10     804218.66
2017-11    1221834.82
2017-12     894992.00
2018-01    1151137.31
2018-02    1027782.86
2018-03    1204543.14
2018-04    1198573.03
2018-05    1192512.47
2018-06    1074756.08
2018-07    1095315.94
2018-08    1038291.04
2018-09        166.46
Freq: M, Name: Revenue, dtype: float64


Top Categories

In [7]:
top = (
    df.groupby("product_category_name")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)
print(top)

product_category_name
beleza_saude                     1485880.29
relogios_presentes               1357478.82
cama_mesa_banho                  1310284.13
esporte_lazer                    1198524.35
informatica_acessorios           1095456.98
                                    ...    
flores                              1598.91
casa_conforto_2                     1194.44
cds_dvds_musicais                    954.99
fashion_roupa_infanto_juvenil        665.36
seguros_e_servicos                   324.51
Name: Revenue, Length: 73, dtype: float64


Step 2: Load the Data

In [3]:
customers = pd.read_csv("olist_customers_dataset.csv")

orders = pd.read_csv("olist_orders_dataset.csv")

items = pd.read_csv("olist_order_items_dataset.csv")

payments = pd.read_csv("olist_order_payments_dataset.csv")

products = pd.read_csv("olist_products_dataset.csv")

reviews = pd.read_csv("olist_order_reviews_dataset.csv")

Step 3: Merge the Tables

In [4]:
df = orders.merge(customers, on="customer_id", how="left")

df = df.merge(items, on="order_id", how="left")

df = df.merge(products, on="product_id", how="left")

df = df.merge(payments, on="order_id", how="left")

df = df.merge(reviews, on="order_id", how="left")

Step 4: Convert Dates

In [8]:
data_columns=[    "order_purchase_timestamp",

    "order_approved_at",

    "order_delivered_carrier_date",

    "order_delivered_customer_date",

    "order_estimated_delivery_date"] 
for col in data_columns:
    df[col]=pd.to_datetime(df[col])

Feature Engineering Revenue

In [11]:
df["Revenue"] = df["price"]

Freight Cost

In [13]:
df["FreightCost"] = df["freight_value"]

Total Sales

In [14]:
df["TotalSales"] = df["price"] + df["freight_value"]

Delivery Days

In [15]:
df["DeliveryDays"] = (

    df["order_delivered_customer_date"]

    - df["order_purchase_timestamp"]

).dt.days

Late Delivery

In [16]:
df["LateDelivery"] = np.where(

    df["order_delivered_customer_date"]

    > df["order_estimated_delivery_date"],

    1,

    0

)

Purchase Month

In [17]:
df["PurchaseMonth"] = (

    df["order_purchase_timestamp"]

    .dt.to_period("M")

)

KPI 1 — Revenue

In [18]:
total_revenue=df["Revenue"].sum()

KPI 2 — Profit

The Olist dataset does not contain product cost, so true profit cannot be calculated.

For a portfolio project, create an estimated cost assumption:

In [21]:
df["EstimatedCost"] = df["Revenue"] * 0.70

Assume a 30% gross margin.

Then:

In [29]:
df["Profit"] = (

    df["Revenue"]

    - df["EstimatedCost"]

)

total_profit = df["Profit"].sum()

print(total_profit)

4282109.895000001


nunique() stands for number of unique values.It counts each different order_id only once.

In [35]:
total_orders = df["order_id"].nunique()

print(total_orders)

99441


KPI 4 — Customers

In [36]:
total_customers = df["customer_unique_id"].nunique()

print(total_customers)

96096


KPI 5 — Average Order Value (AOV)

In [37]:
average_order_value = (

    df.groupby("order_id")["Revenue"]

    .sum()

    .mean()

)

print(average_order_value)

143.53938164338655


KPI 6 — Average Delivery Days

In [38]:
average_delivery = (

    df["DeliveryDays"]

    .mean()

)

print(average_delivery)

12.022588617548953


KPI 7 — Late Delivery %

In [39]:
late_delivery_percentage = (

    df["LateDelivery"]

    .mean()

)*100

print(late_delivery_percentage)

7.611022049134234


KPI 8 — Repeat Customers

In [40]:
customer_orders = (

    df.groupby("customer_unique_id")

    ["order_id"]

    .nunique()

)

In [42]:
repeat_customers = (

    customer_orders > 1

).sum()

print(repeat_customers)

2997


KPI 9 — New Customers

In [43]:
new_customers = (

    customer_orders == 1

).sum()

print(new_customers)

93099


KPI 10 — Monthly Revenue

In [44]:
monthly_revenue = (

    df.groupby("PurchaseMonth")

    ["Revenue"]

    .sum()

)

print(monthly_revenue)

PurchaseMonth
2016-09        267.36
2016-10      51068.92
2016-12         10.90
2017-01     129895.32
2017-02     262013.86
2017-03     398117.44
2017-04     392595.36
2017-05     549226.84
2017-06     456867.47
2017-07     536906.96
2017-08     606026.98
2017-09     665047.38
2017-10     697457.32
2017-11    1055072.10
2017-12     773574.02
2018-01     993701.49
2018-02     889512.29
2018-03    1029589.67
2018-04    1031717.62
2018-05    1032699.42
2018-06     910053.03
2018-07     927401.38
2018-08     884731.52
2018-09        145.00
2018-10          0.00
Freq: M, Name: Revenue, dtype: float64


KPI 11 — Monthly Growth %

In [ ]:
pct_change() computes the percentage change from the previous row.

Formula:

Percentage Change=

(Current Value−Previous Value) / previous value
	​


In [45]:
monthly_growth = (

    monthly_revenue

    .pct_change()

)*100

print(monthly_growth)

PurchaseMonth
2016-09             NaN
2016-10    1.900118e+04
2016-12   -9.997866e+01
2017-01    1.191600e+06
2017-02    1.017115e+02
2017-03    5.194518e+01
2017-04   -1.387048e+00
2017-05    3.989642e+01
2017-06   -1.681625e+01
2017-07    1.751919e+01
2017-08    1.287374e+01
2017-09    9.738906e+00
2017-10    4.873328e+00
2017-11    5.127407e+01
2017-12   -2.668046e+01
2018-01    2.845590e+01
2018-02   -1.048496e+01
2018-03    1.574766e+01
2018-04    2.066794e-01
2018-05    9.516170e-02
2018-06   -1.187629e+01
2018-07    1.906301e+00
2018-08   -4.601013e+00
2018-09   -9.998361e+01
2018-10   -1.000000e+02
Freq: M, Name: Revenue, dtype: float64


KPI 12 — Cancellation %

In [46]:
cancelled = (

    df["order_status"]

    == "canceled"

).sum()

cancellation_rate = (

    cancelled

    /

    df["order_id"].nunique()

)*100

print(cancellation_rate)

0.7542160678191089


KPI 13 — Freight Cost %

In [47]:
freight_percentage = (

    df["FreightCost"].sum()

    /

    df["Revenue"].sum()

)*100

print(freight_percentage)

16.60418607729356


In [48]:
kpis = pd.DataFrame({

    "KPI":[

        "Revenue",

        "Profit",

        "Orders",

        "Customers",

        "Average Order Value",

        "Average Delivery Days",

        "Late Delivery %",

        "Repeat Customers",

        "New Customers",

        "Monthly Growth %",

        "Cancellation %",

        "Freight Cost %"

    ],

    "Value":[

        total_revenue,

        total_profit,

        total_orders,

        total_customers,

        average_order_value,

        average_delivery,

        late_delivery_percentage,

        repeat_customers,

        new_customers,

        monthly_growth.mean(),

        cancellation_rate,

        freight_percentage

    ]

})

print(kpis)

                      KPI         Value
0                 Revenue  1.427370e+07
1                  Profit  4.282110e+06
2                  Orders  9.944100e+04
3               Customers  9.609600e+04
4     Average Order Value  1.435394e+02
5   Average Delivery Days  1.202259e+01
6         Late Delivery %  7.611022e+00
7        Repeat Customers  2.997000e+03
8           New Customers  9.309900e+04
9        Monthly Growth %  5.044024e+04
10         Cancellation %  7.542161e-01
11         Freight Cost %  1.660419e+01


pd.dataframe({kpi:["",""] ,value:[,]})